In [ ]:
!pip install numpy pandas matplotlib folium

In [ ]:
import numpy as np
import random

class CityMap:
    def __init__(self, width=10, height=10):
        self.width = width
        self.height = height

    def random_location(self):
        return (np.random.randint(0, self.width), np.random.randint(0, self.height))

    def distance(self, loc1, loc2):
        return abs(loc1[0] - loc2[0]) + abs(loc1[1] - loc2[1])

In [ ]:
class Driver:
    def __init__(self, driver_id, location):
        self.driver_id = driver_id
        self.location = location
        self.available = True

    def assign_ride(self, new_location):
        self.location = new_location
        self.available = False

    def finish_ride(self):
        self.available = True

In [ ]:
class RideRequest:
    def __init__(self, request_id, pickup, dropoff, request_time):
        self.request_id = request_id
        self.pickup = pickup
        self.dropoff = dropoff
        self.request_time = request_time
        self.assigned_driver = None
        self.price = 0
        self.eta = 0

def generate_requests(city_map, num_requests=5, current_time=0):
    requests = []
    for i in range(num_requests):
        pickup = city_map.random_location()
        dropoff = city_map.random_location()
        requests.append(RideRequest(i, pickup, dropoff, current_time))
    return requests

In [ ]:
def nearest_driver(drivers, pickup_location, city_map):
    best_driver = None
    min_distance = float('inf')
    for driver in drivers:
        if driver.available:
            dist = city_map.distance(driver.location, pickup_location)
            if dist < min_distance:
                min_distance = dist
                best_driver = driver
    return best_driver, min_distance

def weighted_driver_assignment(drivers, pickup_location, city_map, weights=(0.5, 0.5)):
    best_score = float('inf')
    best_driver = None
    for driver in drivers:
        if driver.available:
            dist = city_map.distance(driver.location, pickup_location)
            idle_score = 0 if driver.available else 1
            score = weights[0]*dist + weights[1]*idle_score
            if score < best_score:
                best_score = score
                best_driver = driver
    return best_driver

def calculate_price(distance, base_fare=40, per_km=10, per_min=2, surge=1.0):
    time = distance * 2
    price = base_fare + distance * per_km + time * per_min
    return price * surge

def estimate_eta(distance):
    return distance * 2

In [ ]:
def run_simulation():
    city_map = CityMap(width=10, height=10)

    # Create drivers
    drivers = [Driver(i, city_map.random_location()) for i in range(5)]
    print("Driver locations:", [d.location for d in drivers])

    # Generate ride requests
    requests = generate_requests(city_map, num_requests=5)
    print("\nRide requests:")
    for r in requests:
        print(f"Request {r.request_id}: {r.pickup} -> {r.dropoff}")

    # Assign drivers using weighted algorithm
    for r in requests:
        driver = weighted_driver_assignment(drivers, r.pickup, city_map)
        distance = city_map.distance(r.pickup, r.dropoff)
        r.assigned_driver = driver.driver_id
        r.price = calculate_price(distance, surge=random.choice([1.0, 1.5, 2.0]))
        r.eta = estimate_eta(city_map.distance(driver.location, r.pickup))
        driver.assign_ride(r.dropoff)
        print(f"\nRequest {r.request_id} assigned to Driver {driver.driver_id}")
        print(f"Pickup ETA: {r.eta} min, Price: ₹{r.price:.2f}")

    print("\nFinal Driver locations:", [d.location for d in drivers])

In [ ]:
run_simulation()

Driver locations: [(7, 4), (6, 7), (2, 8), (7, 6), (2, 5)]

Ride requests:
Request 0: (8, 8) -> (7, 9)
Request 1: (1, 9) -> (0, 8)
Request 2: (4, 4) -> (6, 2)
Request 3: (2, 1) -> (7, 9)
Request 4: (5, 1) -> (6, 1)

Request 0 assigned to Driver 1
Pickup ETA: 6 min, Price: ₹102.00

Request 1 assigned to Driver 2
Pickup ETA: 4 min, Price: ₹102.00

Request 2 assigned to Driver 0
Pickup ETA: 6 min, Price: ₹96.00

Request 3 assigned to Driver 4
Pickup ETA: 8 min, Price: ₹222.00

Request 4 assigned to Driver 3
Pickup ETA: 14 min, Price: ₹54.00

Final Driver locations: [(6, 2), (7, 9), (0, 8), (6, 1), (7, 9)]
